<a href="https://colab.research.google.com/github/shaikmujimel-pixel/FDE/blob/main/250200362_Mujimel_Ass3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests

In [ ]:
import re
from bs4 import BeautifulSoup

**Task-1**

In [ ]:
url="https://www.cardekho.com/used-cars+in+chennai/page-2"

In [ ]:
headers = {
    'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML,'
        ' like Gecko) Chrome/120.0.0.0 Safari/537.36'),'Accept-Language': 'en-US,en;q=0.9',
}

In [ ]:
response = requests.get(url, headers=headers)
print('Status Code:', response.status_code)

Status Code: 200


In [ ]:
soup = BeautifulSoup(response.content, 'html.parser')

In [ ]:
car_list = []

for heading in soup.find_all(['h2', 'h3']):
  title = heading.get_text(' ', strip=True)

  # Ensure the heading is a car listing (contains a 4-digit year)
  year = re.search(r'\b(19|20)\d{2}\b', title)
  if not year:
    continue

  manufacturing_year = year.group(0)
  # Remove the year from the title to isolate the model name
  model_name = re.sub(r'\b(19|20)\d{2}\b', '', title).strip()

  # Walk up parent containers until the full car card is captured
  card = heading
  card_text = ''
  for _ in range(7):
    if card.parent is None:
      break
    card = card.parent
    card_text = card.get_text(' ', strip=True)

    if (
        re.search(r'₹\s*[\d,.]+\s*(?:L|Cr|Lakh)', card_text, re.I)
        and re.search(r'[\d,]+\s*kms?', card_text, re.I)
        and re.search(r'\b(Manual|Automatic)\b', card_text, re.I)
    ):
      break

  # 1. Extract Price (ignores "Save ₹XXK" badges by matching Lakhs/Crores first)
  prices = re.findall(r'₹\s*[\d,.]+\s*(?:L|Cr|Lakh)\b', card_text, re.I)
  if not prices:
    prices = re.findall(r'₹\s*[\d,.]+\s*K\b', card_text, re.I)
  price = prices[-1].strip() if prices else None

  # 2. Extract Used in Kilometers
  km_match = re.search(r'[\d,]+\s*kms?', card_text, re.I)
  kms = km_match.group().strip() if km_match else None

  # 3. Extract Address (captures locality + Chennai, including multi-part street addresses)
  location_match = re.search(r'([A-Za-z0-9\s.,&-]+,\s*Chennai)', card_text, re.I)
  if location_match:
    # Clean up any trailing button text like "View Seller Details" before the locality
    raw_address = location_match.group(1).strip()
    address = re.split(r'View Seller Details|Make Offer|Insured till.*',raw_address,
                       flags=re.I,)[-1].strip()
  else:
    address = 'Chennai'

  row = {
      'Title': title,
      'Price': price,
      'Manufacturing year': manufacturing_year,
      'Used in kilometers': kms,
      'Model name': model_name,
      'Address': address,
  }

  # Avoid duplicates and invalid cards
  if kms and price and title not in [x['Title'] for x in car_list]:
    car_list.append(row)

print('Cars extracted:', len(car_list))

Cars extracted: 18


In [ ]:
df = pd.DataFrame(car_list)

In [ ]:
print('Shape:', df.shape)
df.head(20)

Shape: (18, 6)


,Title,Price,Manufacturing year,Used in kilometers,Model name,Address
0,2008 Toyota Innova,₹7.50 L,2008,"1,00,000kms",Toyota Innova,Chennai
1,2016 Hyundai i20,₹4.04 L,2016,"1,10,618kms",Hyundai i20,Chennai
2,2015 Hyundai Grand i10,₹4.12 L,2015,"67,162kms",Hyundai Grand i10,Chennai
3,2018 Hyundai Grand i10,₹4.65 L,2018,"55,266kms",Hyundai Grand i10,Chennai
4,2021 Honda WR-V,₹6.52 L,2021,"50,000kms",Honda WR-V,Chennai
5,2018 Tata Nexon,₹5.64 L,2018,"1,00,559kms",Tata Nexon,Chennai
6,2014 Hyundai Grand i10,₹3.95 L,2014,"69,823kms",Hyundai Grand i10,Chennai
7,2020 Maruti Suzuki Ciaz,₹7.64 L,2020,"29,486kms",Maruti Suzuki Ciaz,Chennai
8,2016 Honda City,₹5.95 L,2016,"55,000kms",Honda City,Chennai
9,2024 Kia Carens,₹17.33 L,2024,"24,477kms",Kia Carens,Chennai


In [ ]:
df.to_csv('cardekho_chennai_data.csv')
print('Data extracted and saved successfully!')

Data extracted and saved successfully!
